# Ollama `qwen3:8b` trên Kaggle cho dịch vụ P3-05/P3-06

Notebook này chạy Ollama với `qwen3:8b` (model chốt ở P3-03) trên GPU của Kaggle, rồi mở đường hầm `cloudflared` để dịch vụ trên máy local gọi tới.

**Cài đặt notebook:** *Accelerator* = **GPU T4 x2** (hoặc P100); *Internet* = **On** (cần xác minh số điện thoại cho tài khoản Kaggle).

Chạy lần lượt các ô. Ô 3 in ra một địa chỉ dạng `https://...trycloudflare.com`. Trên máy local, khởi động lại dịch vụ với địa chỉ đó (xem `moodle/README.md`, mục "LLM trên Kaggle").

Đường hầm không có mật khẩu: ai biết địa chỉ đều gọi được model. Chỉ bật trong lúc demo, xong thì *Stop session*.

In [ ]:
# 1. Cài và khởi động Ollama
import os, subprocess, time, urllib.request
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
env = dict(os.environ, OLLAMA_HOST="0.0.0.0:11434", OLLAMA_KEEP_ALIVE="-1")
ollama = subprocess.Popen(["ollama", "serve"], env=env,
                          stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
print("ollama is up")

In [ ]:
# 2. Tải model (khoảng 5,2 GB) và nạp sẵn lên GPU
subprocess.run(["ollama", "pull", "qwen3:8b"], check=True)
import json
req = urllib.request.Request("http://127.0.0.1:11434/api/chat", method="POST",
    headers={"Content-Type": "application/json"},
    data=json.dumps({"model": "qwen3:8b", "think": False, "stream": False,
                     "messages": [{"role": "user", "content": "Reply with the number 4."}]}).encode())
t = time.time()
print(json.loads(urllib.request.urlopen(req, timeout=600).read())["message"]["content"], f"({time.time()-t:.1f}s)")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"])

In [ ]:
# 3. Mở đường hầm cloudflared và in địa chỉ
import re
subprocess.run("wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /kaggle/working/cloudflared", shell=True, check=True)
log = open("/kaggle/working/cloudflared.log", "w")
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate",
                           "--url", "http://127.0.0.1:11434", "--http-host-header", "localhost:11434"],
                          stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/cloudflared.log").read())
    if m:
        url = m.group(0); break
print("OLLAMA_BASE_URL =", url)

In [ ]:
# 4. Kiểm tra qua đường hầm (đợi vài giây nếu lần đầu báo lỗi DNS)
time.sleep(5)
print(urllib.request.urlopen(url + "/api/tags", timeout=30).read()[:200])

In [ ]:
# 5. Giữ phiên chạy; mỗi 5 phút in số request đã phục vụ. Dừng bằng Interrupt / Stop session.
while True:
    time.sleep(300)
    served = open("/kaggle/working/ollama.log").read().count("/api/chat")
    print(time.strftime("%H:%M:%S"), "chat requests:", served, "| ollama alive:", ollama.poll() is None,
          "| tunnel alive:", tunnel.poll() is None, flush=True)